# Lab 08: Model Evaluation

## Overview
In this lab you will evaluate foundation model outputs using Bedrock's automatic evaluation, custom metrics, and hallucination detection. You will build an evaluation dataset with known-correct reference answers, run Bedrock's managed evaluation job, implement custom scoring metrics (ROUGE-1 and embedding similarity), and detect hallucinations using a grounding check. Finally, you will compare Claude, Llama, and Titan on the same test set using a comparison dashboard.

## Learning Objectives
- Build evaluation datasets with reference answers and upload to S3 as JSONL
- Use Bedrock automatic evaluation to assess model outputs at scale
- Implement custom metrics: ROUGE-1 (unigram overlap) and embedding similarity (BERTScore proxy)
- Detect hallucinations by checking whether model answers are grounded in provided context
- Build a comparison dashboard that ranks models across all metrics

## Exam Domain
**Optimizing Performance & Inference (24%)** — This lab covers Task 3.1 (evaluate and improve model outputs), focusing on automatic and custom evaluation methods, hallucination detection, and systematic model comparison for production readiness.

## Architecture Diagram
![Lab 08 Architecture](../assets/diagrams/lab-08-model-evaluation.png)

In [ ]:
%pip install boto3 numpy --quiet

In [ ]:
import boto3, json, os, time
import numpy as np

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")
bedrock = session.client("bedrock")
s3 = session.client("s3")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"

print(f"Account:      {ACCOUNT_ID}")
print(f"S3 bucket:    {BUCKET}")
print(f"Bedrock role: {BEDROCK_ROLE_ARN}")

In [ ]:
# Helper: unified Converse API wrapper for any Bedrock text model
def invoke_converse(model_id, prompt, max_tokens=256, temperature=0.7):
    response = bedrock_runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature}
    )
    return response["output"]["message"]["content"][0]["text"]


# Helper: Titan Embeddings V2
def get_embedding(text):
    """Get embedding from Titan Embeddings V2."""
    response = bedrock_runtime.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        contentType="application/json",
        body=json.dumps({"inputText": text})
    )
    return json.loads(response["body"].read())["embedding"]


# Models to evaluate
MODELS = [
    ("anthropic.claude-3-5-sonnet-20241022-v2:0", "Claude 3.5 Sonnet"),
    ("meta.llama3-8b-instruct-v1:0", "Llama 3 8B"),
    ("amazon.titan-text-express-v1", "Titan Text Express"),
]

print("Helper functions defined.")
print(f"Models to evaluate: {[name for _, name in MODELS]}")

## A. Building an Evaluation Dataset

Every model evaluation starts with a **test set** — a collection of questions paired with known-correct reference answers. The quality of your evaluation is only as good as your test data.

For Bedrock's automatic evaluation, the dataset must be uploaded to S3 in **JSONL format** (one JSON object per line). Each record contains:
- **prompt** — the question or instruction sent to the model
- **referenceResponse** — the ground-truth answer to compare against

We will create 12 Q&A pairs about AWS services with factual, concise reference answers. These cover a range of services to test general AWS knowledge across models.

In [ ]:
# Evaluation dataset: Q&A pairs with reference answers about AWS services
eval_dataset = [
    {
        "prompt": "What is Amazon S3 and what is it used for?",
        "referenceResponse": "Amazon S3 (Simple Storage Service) is an object storage service that provides scalable, durable, and highly available storage. It is used for storing and retrieving any amount of data, hosting static websites, data lakes, backups, and serving content for applications."
    },
    {
        "prompt": "What is AWS Lambda?",
        "referenceResponse": "AWS Lambda is a serverless compute service that runs code in response to events without requiring you to provision or manage servers. You pay only for the compute time consumed, and it automatically scales from a few requests per day to thousands per second."
    },
    {
        "prompt": "What is Amazon DynamoDB?",
        "referenceResponse": "Amazon DynamoDB is a fully managed NoSQL database service that provides fast and predictable performance with seamless scalability. It supports key-value and document data models, offers single-digit millisecond latency, and includes built-in security, backup, and in-memory caching."
    },
    {
        "prompt": "What is Amazon Bedrock?",
        "referenceResponse": "Amazon Bedrock is a fully managed service that provides access to foundation models from leading AI companies through a single API. It allows you to choose from models by Anthropic, Meta, Amazon, and others, and provides tools for customization, evaluation, and building generative AI applications."
    },
    {
        "prompt": "What is Amazon SageMaker used for?",
        "referenceResponse": "Amazon SageMaker is a fully managed machine learning service that enables developers and data scientists to build, train, and deploy ML models. It provides integrated Jupyter notebooks, built-in algorithms, one-click training, and managed hosting for model inference."
    },
    {
        "prompt": "What are the three types of load balancers in AWS?",
        "referenceResponse": "AWS offers three types of load balancers through Elastic Load Balancing: Application Load Balancer (ALB) for HTTP/HTTPS traffic at Layer 7, Network Load Balancer (NLB) for TCP/UDP traffic at Layer 4 with ultra-low latency, and Gateway Load Balancer (GWLB) for deploying and managing third-party virtual network appliances."
    },
    {
        "prompt": "What is Amazon VPC?",
        "referenceResponse": "Amazon VPC (Virtual Private Cloud) is a service that lets you provision a logically isolated section of the AWS Cloud where you can launch resources in a virtual network you define. You have complete control over your networking environment, including IP address ranges, subnets, route tables, and network gateways."
    },
    {
        "prompt": "What is the difference between Amazon RDS and Amazon Aurora?",
        "referenceResponse": "Amazon RDS is a managed relational database service supporting multiple engines (MySQL, PostgreSQL, MariaDB, Oracle, SQL Server). Amazon Aurora is a MySQL and PostgreSQL-compatible relational database built for the cloud that provides up to five times the throughput of standard MySQL and three times the throughput of standard PostgreSQL, with greater scalability and availability than standard RDS."
    },
    {
        "prompt": "What is Amazon CloudFront?",
        "referenceResponse": "Amazon CloudFront is a content delivery network (CDN) service that securely delivers data, videos, applications, and APIs to users with low latency and high transfer speeds. It uses a global network of edge locations to cache content close to end users."
    },
    {
        "prompt": "What is AWS IAM?",
        "referenceResponse": "AWS IAM (Identity and Access Management) is a service that enables you to securely control access to AWS resources. It lets you create and manage users, groups, and roles, and use permissions policies to allow or deny access to specific AWS services and resources."
    },
    {
        "prompt": "What is Amazon ECS?",
        "referenceResponse": "Amazon ECS (Elastic Container Service) is a fully managed container orchestration service that makes it easy to deploy, manage, and scale containerized applications. It supports Docker containers and allows you to run applications on a managed cluster of EC2 instances or on AWS Fargate for serverless containers."
    },
    {
        "prompt": "What is Amazon Kinesis used for?",
        "referenceResponse": "Amazon Kinesis is a platform for streaming data on AWS, enabling you to collect, process, and analyze real-time data streams. It includes Kinesis Data Streams for custom stream processing, Kinesis Data Firehose for loading streams into data stores, and Kinesis Data Analytics for analyzing streams with SQL or Apache Flink."
    },
]

print(f"Evaluation dataset: {len(eval_dataset)} Q&A pairs")
print(f"\nSample entry:")
print(f"  Prompt:    {eval_dataset[0]['prompt']}")
print(f"  Reference: {eval_dataset[0]['referenceResponse'][:80]}...")

In [ ]:
# Upload evaluation dataset to S3 as JSONL (one JSON object per line)
jsonl_content = "\n".join(json.dumps(record) for record in eval_dataset)
eval_key = "evaluation/eval-dataset.jsonl"

s3.put_object(
    Bucket=BUCKET,
    Key=eval_key,
    Body=jsonl_content.encode("utf-8"),
    ContentType="application/jsonl"
)

print(f"Uploaded {len(eval_dataset)} records to s3://{BUCKET}/{eval_key}")
print(f"File size: {len(jsonl_content):,} bytes")

## B. Automatic Evaluation with Bedrock

Amazon Bedrock provides a **managed evaluation service** that runs evaluation jobs asynchronously. You submit a dataset, choose the model(s) to evaluate, and select evaluation metrics — Bedrock handles the rest.

### Evaluation Job Types

| Type | Description | Use Case |
|------|-------------|----------|
| **Automatic** | Bedrock computes metrics (accuracy, robustness, toxicity) using built-in algorithms | Rapid, repeatable scoring at scale |
| **Human** | Human reviewers rate model outputs on criteria you define | Subjective quality, tone, helpfulness |
| **Model-as-Judge** | A foundation model (e.g., Claude) evaluates another model's outputs | Scalable alternative to human eval |

The `create_evaluation_job` API creates an asynchronous job. The job reads your dataset from S3, invokes the target model for each prompt, compares outputs against reference answers, and writes results back to S3.

> **Note:** Evaluation jobs can take several minutes to complete. The cell below shows the API call structure and polls for completion.

In [ ]:
# Create an automatic evaluation job using Bedrock
# This evaluates Claude 3.5 Sonnet on our Q&A dataset using built-in accuracy metrics.

JOB_NAME = f"genai-lab-eval-{int(time.time())}"
OUTPUT_PREFIX = "evaluation/results/"

try:
    response = bedrock.create_evaluation_job(
        jobName=JOB_NAME,
        jobDescription="Lab 08 — evaluate Claude on AWS Q&A dataset",
        roleArn=BEDROCK_ROLE_ARN,
        evaluationConfig={
            "automated": {
                "datasetMetricConfigs": [
                    {
                        "taskType": "QuestionAndAnswer",
                        "metricNames": ["Accuracy", "Robustness"],
                        "dataset": {
                            "name": "aws-qa-eval",
                            "datasetLocation": {
                                "s3Uri": f"s3://{BUCKET}/{eval_key}"
                            }
                        }
                    }
                ]
            }
        },
        inferenceConfig={
            "models": [
                {
                    "bedrockModel": {
                        "modelIdentifier": "anthropic.claude-3-5-sonnet-20241022-v2:0",
                        "inferenceParams": json.dumps({
                            "maxTokens": 256,
                            "temperature": 0
                        })
                    }
                }
            ]
        },
        outputDataConfig={
            "s3Uri": f"s3://{BUCKET}/{OUTPUT_PREFIX}"
        }
    )

    EVAL_JOB_ARN = response["jobArn"]
    print(f"Evaluation job created: {JOB_NAME}")
    print(f"Job ARN: {EVAL_JOB_ARN}")

except Exception as e:
    EVAL_JOB_ARN = None
    print(f"Could not create evaluation job: {e}")
    print("This is expected if your IAM role lacks bedrock:CreateEvaluationJob permission.")
    print("We will use custom metrics in Sections C-E instead.")

In [ ]:
# Poll for evaluation job completion (may take 5-15 minutes)
if EVAL_JOB_ARN:
    print(f"Polling evaluation job: {JOB_NAME}")
    print("This may take 5-15 minutes...\n")

    while True:
        job = bedrock.get_evaluation_job(jobIdentifier=EVAL_JOB_ARN)
        status = job["status"]
        print(f"  Status: {status}")

        if status in ("Completed", "Failed", "Stopped"):
            break
        time.sleep(30)

    if status == "Completed":
        print(f"\nJob completed successfully.")
        print(f"Results written to: s3://{BUCKET}/{OUTPUT_PREFIX}")

        # Retrieve and display results
        result_objects = s3.list_objects_v2(
            Bucket=BUCKET, Prefix=OUTPUT_PREFIX
        ).get("Contents", [])
        for obj in result_objects:
            print(f"  - {obj['Key']} ({obj['Size']:,} bytes)")
    else:
        print(f"\nJob ended with status: {status}")
        if "failureMessages" in job:
            for msg in job["failureMessages"]:
                print(f"  Error: {msg}")
else:
    print("Skipping — no evaluation job was created.")
    print("Proceeding to custom metrics in the following sections.")

## C. Custom Metrics: ROUGE Score

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures the overlap between a model's output (hypothesis) and a reference answer. It was originally designed for summarization evaluation but is widely used for any text generation task.

**ROUGE-1** specifically measures unigram (single-word) overlap. It computes:
- **Precision** — what fraction of the model's words also appear in the reference
- **Recall** — what fraction of the reference's words also appear in the model's output
- **F1** — the harmonic mean of precision and recall

A ROUGE-1 F1 score of 1.0 means perfect word overlap; 0.0 means no words in common. For Q&A evaluation, scores above 0.4 typically indicate reasonable coverage of the reference answer.

We implement ROUGE-1 from scratch below — no external libraries needed.

In [ ]:
def rouge_1(reference, hypothesis):
    """Compute ROUGE-1 F1 score (unigram overlap) between reference and hypothesis."""
    ref_tokens = set(reference.lower().split())
    hyp_tokens = set(hypothesis.lower().split())
    if not ref_tokens:
        return 0.0
    overlap = ref_tokens & hyp_tokens
    precision = len(overlap) / len(hyp_tokens) if hyp_tokens else 0
    recall = len(overlap) / len(ref_tokens)
    if precision + recall == 0:
        return 0.0
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


# Quick test
ref = "Amazon S3 is an object storage service for scalable data storage."
hyp = "Amazon S3 provides scalable object storage for any amount of data."
score = rouge_1(ref, hyp)
print(f"Reference:  {ref}")
print(f"Hypothesis: {hyp}")
print(f"ROUGE-1 F1: {score:.4f}")

In [ ]:
# Run ROUGE-1 evaluation across all models and all test questions
rouge_scores = {}

for model_id, model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    scores = []
    for i, item in enumerate(eval_dataset):
        try:
            answer = invoke_converse(model_id, item["prompt"], max_tokens=256, temperature=0)
            score = rouge_1(item["referenceResponse"], answer)
            scores.append(score)
            print(f"  Q{i+1:2d}: ROUGE-1 = {score:.4f} | {item['prompt'][:50]}...")
        except Exception as e:
            print(f"  Q{i+1:2d}: ERROR — {e}")
            scores.append(0.0)

    avg_score = np.mean(scores)
    rouge_scores[model_name] = {"scores": scores, "avg": avg_score}
    print(f"\n  Average ROUGE-1: {avg_score:.4f}")

In [ ]:
# ROUGE-1 comparison table
print(f"\n{'Model':<25} {'Avg ROUGE-1':>12}")
print("-" * 37)
for model_name, data in rouge_scores.items():
    print(f"{model_name:<25} {data['avg']:>12.4f}")

## D. Custom Metrics: Embedding Similarity (BERTScore Proxy)

ROUGE only measures surface-level word overlap. Two sentences can convey the same meaning with completely different words and score poorly on ROUGE. **BERTScore** addresses this by comparing text using contextual embeddings, capturing semantic similarity rather than lexical overlap.

We use **Titan Embeddings V2** as a proxy for BERTScore: embed both the reference answer and the model output, then compute cosine similarity between the two vectors. A similarity of 1.0 means the texts are semantically identical; 0.0 means completely unrelated.

This approach is especially useful when a model gives a correct answer using different phrasing than the reference.

In [ ]:
def embedding_similarity(text_a, text_b):
    """Compute cosine similarity between Titan embeddings of two texts."""
    emb_a = get_embedding(text_a)
    emb_b = get_embedding(text_b)
    return np.dot(emb_a, emb_b) / (np.linalg.norm(emb_a) * np.linalg.norm(emb_b))


# Quick test: semantically similar vs. different texts
sim_same = embedding_similarity(
    "Amazon S3 is an object storage service.",
    "S3 provides scalable object storage in the cloud."
)
sim_diff = embedding_similarity(
    "Amazon S3 is an object storage service.",
    "AWS Lambda runs serverless functions."
)
print(f"Similar meaning: {sim_same:.4f}")
print(f"Different topics: {sim_diff:.4f}")

In [ ]:
# Run embedding similarity evaluation across all models
# We reuse the model outputs from Section C to avoid redundant API calls.
# First, collect all model answers.

model_answers = {}

for model_id, model_name in MODELS:
    print(f"\nCollecting answers from: {model_name}")
    answers = []
    for i, item in enumerate(eval_dataset):
        try:
            answer = invoke_converse(model_id, item["prompt"], max_tokens=256, temperature=0)
            answers.append(answer)
        except Exception as e:
            print(f"  Q{i+1}: ERROR — {e}")
            answers.append("")
    model_answers[model_name] = answers
    print(f"  Collected {len(answers)} answers")

print(f"\nAll model answers collected.")

In [ ]:
# Compute embedding similarity for each model's answers vs. reference answers
similarity_scores = {}

for model_name, answers in model_answers.items():
    print(f"\n{'='*60}")
    print(f"Embedding similarity: {model_name}")
    print(f"{'='*60}")

    scores = []
    for i, (item, answer) in enumerate(zip(eval_dataset, answers)):
        if answer:
            sim = embedding_similarity(item["referenceResponse"], answer)
        else:
            sim = 0.0
        scores.append(sim)
        print(f"  Q{i+1:2d}: Similarity = {sim:.4f} | {item['prompt'][:50]}...")

    avg_sim = np.mean(scores)
    similarity_scores[model_name] = {"scores": scores, "avg": avg_sim}
    print(f"\n  Average similarity: {avg_sim:.4f}")

# Comparison table
print(f"\n\n{'Model':<25} {'Avg Emb Similarity':>18}")
print("-" * 43)
for model_name, data in similarity_scores.items():
    print(f"{model_name:<25} {data['avg']:>18.4f}")

## E. Hallucination Detection

A **hallucination** occurs when a model generates information that is not supported by the provided context — it "makes things up" that sound plausible but are factually incorrect or ungrounded.

Hallucination detection is critical for production GenAI applications, especially in regulated industries (healthcare, finance, legal) where incorrect information can have serious consequences.

Our approach uses **model-as-judge**: we give Claude a specific context and an answer, then ask it to determine whether the answer is fully **grounded** in the context. This is a practical, scalable technique that avoids the need for human reviewers on every output.

Grounding levels:
- **GROUNDED** — all claims in the answer are supported by the context
- **PARTIALLY_GROUNDED** — some claims are supported, but the answer adds information not in the context
- **NOT_GROUNDED** — the answer contains significant information not found in the context

In [ ]:
def check_grounding(context, answer):
    """Use Claude as a judge to check if an answer is grounded in the provided context."""
    prompt = f"""Given the following context and answer, determine if the answer is fully grounded in the context.

Context: {context}

Answer: {answer}

Respond with ONLY one of: GROUNDED, PARTIALLY_GROUNDED, NOT_GROUNDED
Then explain why in one sentence."""
    return invoke_converse("anthropic.claude-3-5-sonnet-20241022-v2:0", prompt, temperature=0.0)


# Test with a grounded answer
context = "Amazon S3 provides object storage with 99.999999999% durability. It stores data as objects in buckets."
grounded_answer = "S3 stores data as objects in buckets and offers eleven nines of durability."
hallucinated_answer = "S3 stores data as objects in buckets, offers eleven nines of durability, and includes built-in machine learning for automatic data classification."

print("--- Grounded answer ---")
print(f"Context: {context}")
print(f"Answer:  {grounded_answer}")
result = check_grounding(context, grounded_answer)
print(f"Verdict: {result}\n")

print("--- Hallucinated answer ---")
print(f"Context: {context}")
print(f"Answer:  {hallucinated_answer}")
result = check_grounding(context, hallucinated_answer)
print(f"Verdict: {result}")

In [ ]:
# Hallucination detection across all models
# We ask each model a question WITH a specific context, then check if its answer
# stays grounded in that context.

grounding_contexts = [
    {
        "context": "Amazon DynamoDB is a key-value and document database that delivers single-digit millisecond performance at any scale. It supports both key-value and document data models. DynamoDB handles more than 10 trillion requests per day and supports peaks of more than 20 million requests per second.",
        "question": "Based on the context provided, describe Amazon DynamoDB."
    },
    {
        "context": "AWS Lambda lets you run code without provisioning or managing servers. You pay only for the compute time you consume. Lambda supports Python, Node.js, Java, Go, Ruby, and .NET. The maximum execution time for a Lambda function is 15 minutes.",
        "question": "Based on the context provided, describe AWS Lambda."
    },
    {
        "context": "Amazon Bedrock is a fully managed service for foundation models. It provides access to models from Anthropic, Meta, Amazon, Cohere, and Stability AI. Bedrock supports text generation, embeddings, and image generation use cases.",
        "question": "Based on the context provided, describe Amazon Bedrock."
    },
    {
        "context": "Amazon CloudWatch monitors AWS resources and applications. It collects metrics, logs, and events. CloudWatch Alarms can trigger notifications when metrics cross thresholds. CloudWatch Logs Insights enables interactive log analysis.",
        "question": "Based on the context provided, describe Amazon CloudWatch."
    },
    {
        "context": "Amazon SQS is a fully managed message queuing service. It offers two queue types: Standard (best-effort ordering, at-least-once delivery) and FIFO (exactly-once processing, strict ordering). The maximum message size is 256 KB.",
        "question": "Based on the context provided, describe Amazon SQS."
    },
]

grounding_results = {}

for model_id, model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Grounding check: {model_name}")
    print(f"{'='*60}")

    verdicts = []
    for i, item in enumerate(grounding_contexts):
        # Ask the model to answer based on the context
        prompt = f"Context: {item['context']}\n\nQuestion: {item['question']}\n\nAnswer based ONLY on the provided context."
        try:
            answer = invoke_converse(model_id, prompt, max_tokens=256, temperature=0)
            # Use Claude as judge
            verdict = check_grounding(item["context"], answer)
            # Extract the grounding label from the first line
            label = verdict.strip().split("\n")[0].strip()
            verdicts.append(label)
            print(f"  Q{i+1}: {label}")
        except Exception as e:
            print(f"  Q{i+1}: ERROR — {e}")
            verdicts.append("ERROR")

    # Calculate grounded percentage
    grounded_count = sum(1 for v in verdicts if "GROUNDED" == v.upper() or v.upper().startswith("GROUNDED"))
    grounded_pct = grounded_count / len(verdicts) if verdicts else 0
    grounding_results[model_name] = {"verdicts": verdicts, "grounded_pct": grounded_pct}
    print(f"\n  Grounded: {grounded_count}/{len(verdicts)} ({grounded_pct:.0%})")

## F. Comparison Dashboard

Now we bring all metrics together into a single comparison table. This is the kind of dashboard you would build in production to decide which model to deploy for a specific use case.

Each model is scored on three dimensions:
- **ROUGE-1** — lexical overlap with reference answers (Section C)
- **Embedding Similarity** — semantic similarity to reference answers (Section D)
- **Grounded %** — percentage of context-based answers that stayed grounded (Section E)

No single metric tells the whole story. A model might score high on embedding similarity (correct meaning) but low on ROUGE (different words), or score high on both but still hallucinate when given limited context.

In [ ]:
# Comparison dashboard: all models across all metrics
print("=" * 60)
print("MODEL EVALUATION DASHBOARD")
print("=" * 60)
print(f"\n{'Model':<25} {'ROUGE-1':>10} {'Emb Sim':>10} {'Grounded':>10}")
print("-" * 55)

results = {}
for model_name in [name for _, name in MODELS]:
    rouge_avg = rouge_scores.get(model_name, {}).get("avg", 0.0)
    sim_avg = similarity_scores.get(model_name, {}).get("avg", 0.0)
    grounded_pct = grounding_results.get(model_name, {}).get("grounded_pct", 0.0)

    results[model_name] = {
        "rouge": rouge_avg,
        "similarity": sim_avg,
        "grounded_pct": grounded_pct
    }
    print(f"{model_name:<25} {rouge_avg:>10.4f} {sim_avg:>10.4f} {grounded_pct:>9.0%}")

print("-" * 55)
print("\nInterpretation:")
print("  ROUGE-1:  Higher = more word overlap with reference answers")
print("  Emb Sim:  Higher = more semantically similar to reference answers")
print("  Grounded: Higher = fewer hallucinations when given limited context")

In [ ]:
# Per-question breakdown for the best-performing model
best_model = max(results.items(), key=lambda x: x[1]["similarity"])[0]
print(f"Per-question breakdown for: {best_model}\n")
print(f"{'Q#':<5} {'ROUGE-1':>10} {'Emb Sim':>10} {'Question':<45}")
print("-" * 70)

for i, item in enumerate(eval_dataset):
    r = rouge_scores[best_model]["scores"][i]
    s = similarity_scores[best_model]["scores"][i]
    q = item["prompt"][:42] + "..." if len(item["prompt"]) > 42 else item["prompt"]
    print(f"Q{i+1:<4} {r:>10.4f} {s:>10.4f} {q:<45}")

## Cleanup

Delete the evaluation dataset and any results from S3. If a Bedrock evaluation job was created, stop it if still running.

In [ ]:
# Clean up S3 evaluation data
try:
    # Delete evaluation dataset
    s3.delete_object(Bucket=BUCKET, Key=eval_key)
    print(f"Deleted: s3://{BUCKET}/{eval_key}")

    # Delete any evaluation results
    result_objects = s3.list_objects_v2(
        Bucket=BUCKET, Prefix="evaluation/"
    ).get("Contents", [])
    for obj in result_objects:
        s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
        print(f"Deleted: s3://{BUCKET}/{obj['Key']}")
except Exception as e:
    print(f"Cleanup error: {e}")

# Stop evaluation job if still running
if EVAL_JOB_ARN:
    try:
        job = bedrock.get_evaluation_job(jobIdentifier=EVAL_JOB_ARN)
        if job["status"] in ("InProgress", "Submitted"):
            bedrock.stop_evaluation_job(jobIdentifier=EVAL_JOB_ARN)
            print(f"Stopped evaluation job: {JOB_NAME}")
        else:
            print(f"Evaluation job already {job['status']}")
    except Exception as e:
        print(f"Could not stop evaluation job: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. **No single metric captures model quality** — ROUGE measures lexical overlap, embedding similarity captures semantic meaning, and grounding checks detect hallucinations; production evaluations should combine multiple metrics for a complete picture
2. **Bedrock's automatic evaluation runs at scale without custom code** — the `create_evaluation_job` API handles dataset ingestion, model invocation, scoring, and result storage as a managed asynchronous job, ideal for repeatable benchmarking
3. **Custom metrics give you control over what matters for your use case** — ROUGE and embedding similarity are easy to implement without external libraries, and you can tailor scoring to domain-specific requirements
4. **Hallucination detection requires grounding checks, not just accuracy scores** — a model can score high on ROUGE and similarity while still adding fabricated information; model-as-judge grounding checks catch this failure mode
5. **Model comparison dashboards drive informed selection decisions** — different models excel on different metrics, and the right choice depends on whether your use case prioritizes accuracy, semantic coverage, or factual grounding

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Evaluation Job | A managed Bedrock service that asynchronously evaluates model outputs against a test dataset, computing built-in metrics like accuracy, robustness, and toxicity |
| ROUGE | Recall-Oriented Understudy for Gisting Evaluation — a family of metrics that measure word overlap between generated text and reference text, commonly used for summarization and Q&A evaluation |
| BERTScore | An evaluation metric that uses contextual embeddings to compare generated text with reference text at the semantic level, capturing meaning beyond surface-level word matching |
| Embedding Similarity | Cosine similarity between vector embeddings of two texts, used as a proxy for BERTScore when a dedicated BERTScore library is unavailable — higher values indicate closer semantic alignment |
| Hallucination | When a model generates information that is not supported by the provided context or facts, producing plausible-sounding but incorrect or fabricated content |
| Grounding | The degree to which a model's output is supported by and traceable to the provided source context, with fully grounded answers containing only information present in the context |
| Human Evaluation | A Bedrock evaluation job type where human reviewers rate model outputs on custom criteria (helpfulness, tone, accuracy), providing subjective quality assessment that automated metrics cannot capture |

## Exam Preparation — Common Exam Question Patterns

**Q: What are the evaluation methods available in Amazon Bedrock?**
A: Bedrock supports three evaluation methods: (1) Automatic evaluation, which computes built-in metrics like accuracy, robustness, and toxicity using algorithms; (2) Human evaluation, where human reviewers rate outputs against custom criteria; and (3) Model-as-judge evaluation, where a foundation model evaluates another model's outputs. Automatic evaluation is best for repeatable benchmarking at scale, while human evaluation captures subjective quality dimensions.

**Q: How does ROUGE differ from BERTScore for evaluating model outputs?**
A: ROUGE measures surface-level lexical overlap (word matching) between generated text and reference text. BERTScore uses contextual embeddings to measure semantic similarity, capturing meaning even when different words are used. ROUGE is fast and interpretable but misses paraphrases; BERTScore correlates better with human judgment but requires embedding computation. Use ROUGE for quick checks and BERTScore (or embedding similarity) when semantic accuracy matters.

**Q: How would you detect hallucinations in a GenAI application?**
A: Hallucination detection checks whether model outputs are grounded in the provided source material. Common approaches include: (1) Model-as-judge, where a capable model evaluates whether each claim in the output is supported by the context; (2) Entailment classification, which determines if the context logically entails the generated statements; and (3) Fact extraction and verification, which extracts individual facts from the output and checks each against the source. For production systems, combine automated grounding checks with human review of edge cases.

**Q: When should you use Bedrock's automatic evaluation vs. custom evaluation metrics?**
A: Use Bedrock's automatic evaluation when you need standardized, repeatable benchmarks across models and want metrics like accuracy, robustness, and toxicity computed without custom code. Use custom metrics when you need domain-specific scoring (e.g., medical terminology accuracy), when you want to combine multiple evaluation dimensions into a single dashboard, or when you need real-time evaluation integrated into your inference pipeline rather than batch evaluation jobs.

**Q: What factors should you consider when comparing foundation models for a production use case?**
A: Compare models across multiple dimensions: (1) Quality metrics — ROUGE, BERTScore, and task-specific accuracy on your evaluation dataset; (2) Grounding and safety — hallucination rates, toxicity scores, and adherence to provided context; (3) Latency — time to first token and total generation time; (4) Cost — price per input/output token and total cost at your expected volume; and (5) Consistency — variance in output quality across different prompts. No single metric determines the best model; the right choice depends on which dimensions matter most for your specific use case.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Claude 3.5 Sonnet — Section C (ROUGE eval, 12 questions) | 12 calls, ~6K tokens | ~$0.10 |
| Llama 3 8B — Section C (ROUGE eval, 12 questions) | 12 calls, ~6K tokens | ~$0.01 |
| Titan Text Express — Section C (ROUGE eval, 12 questions) | 12 calls, ~6K tokens | ~$0.01 |
| Claude 3.5 Sonnet — Section D (embedding eval, 12 questions) | 12 calls, ~6K tokens | ~$0.10 |
| Llama 3 8B — Section D (embedding eval, 12 questions) | 12 calls, ~6K tokens | ~$0.01 |
| Titan Text Express — Section D (embedding eval, 12 questions) | 12 calls, ~6K tokens | ~$0.01 |
| Titan Embeddings V2 — Section D (similarity, ~72 embeddings) | 72 calls, ~18K tokens | ~$0.01 |
| Claude 3.5 Sonnet — Section E (grounding, 5 questions x 3 models) | 15 calls, ~8K tokens | ~$0.15 |
| Llama 3 8B — Section E (grounding answers, 5 questions) | 5 calls, ~2.5K tokens | < $0.01 |
| Titan Text Express — Section E (grounding answers, 5 questions) | 5 calls, ~2.5K tokens | < $0.01 |
| Claude 3.5 Sonnet — Section E (judge, 15 grounding checks) | 15 calls, ~8K tokens | ~$0.15 |
| Bedrock Evaluation Job — Section B (if created) | 1 job, 12 prompts | ~$1.00 |
| **Total (without Bedrock eval job)** | | **~$0.60** |
| **Total (with Bedrock eval job)** | | **~$2-3** |

The primary cost drivers are the Claude 3.5 Sonnet API calls for generating answers and judging grounding. Llama and Titan calls are significantly cheaper. The Bedrock automatic evaluation job (Section B) is the most expensive single item but is optional and provides managed evaluation at scale. Titan Embeddings calls are negligible.